In [13]:
import pandas as pd

# CSV-Dateien laden
df_seq = pd.read_csv("../../documentation/pdb_sequences_detailed.csv")
df_map = pd.read_csv("../../documentation/clean_entity_mapping.csv")

# Einheitliche PDB-IDs
df_seq["pdb_id"] = df_seq["pdb_id"].str.lower()
df_map["pdb"] = df_map["pdb"].str.lower()

# Nur relevante Spalten: Heavy Chains
df_map = df_map[["pdb", "H_entity_id", "heavy_subclass", "light_subclass"]]

# Typkonvertierung: entity_id als Integer (wenn nicht bereits)
df_seq["entity_id"] = df_seq["entity_id"].astype(int)
df_map = df_map[df_map["H_entity_id"].notna()]
df_map["H_entity_id"] = df_map["H_entity_id"].astype(int)

# Merge der Sequenzen mit den Genfamilien-Infos
merged = pd.merge(
    df_seq,
    df_map,
    how="left",
    left_on=["pdb_id", "entity_id"],
    right_on=["pdb", "H_entity_id"]
)

# Ausgabe speichern
merged.to_csv("pdb_sequences_with_subclasses.csv", index=False)
print(f"[INFO] Merge abgeschlossen. {merged['heavy_subclass'].notna().sum()} Heavy Chains mit Zuordnung.")

[INFO] Merge abgeschlossen. 2589 Heavy Chains mit Zuordnung.


nun habe ich die Datei in eine fasta umgewandelt.

In [14]:
import pandas as pd

def write_heavy_chains_fasta(input_csv="pdb_sequences_with_subclasses.csv", output_fasta="heavy_chains_with_families.fasta"):
    # CSV-Datei laden
    df = pd.read_csv(input_csv)

    # Nur Heavy Chains mit gültiger heavy_subclass auswählen
    df_heavy = df[df["heavy_subclass"].notna()].copy()

    # FASTA schreiben
    with open(output_fasta, "w") as f:
        for _, row in df_heavy.iterrows():
            header = f">{row['pdb']}_entity{row['entity_id']}|{row['chain_ids']}|{row['heavy_subclass']}"
            sequence = row['sequence']
            f.write(f"{header}\n{sequence}\n\n")

    print(f"[DONE] {len(df_heavy)} Heavy Chains gespeichert in {output_fasta}")

# Beispielaufruf
write_heavy_chains_fasta()

[DONE] 2589 Heavy Chains gespeichert in heavy_chains_with_families.fasta


Code für Mulitiple Sequence Alignment, welches Consensus Sequenzen für jede Genfamilien generiert. 

In [22]:
import os
from Bio import SeqIO
from collections import defaultdict
import subprocess

input_fasta = "heavy_chains_with_families.fasta"
output_dir = "consensus_clustalo"
os.makedirs(output_dir, exist_ok=True)

# Gruppieren nach Genfamilien
sequences_by_family = defaultdict(list)
for record in SeqIO.parse(input_fasta, "fasta"):
    try:
        genfamily = record.id.split("|")[-1].strip()
        sequences_by_family[genfamily].append(record)
    except IndexError:
        continue

print(f"[INFO] {len(sequences_by_family)} Genfamilien gefunden.")

# Für jede Familie MSA mit Clustal Omega
for family, records in sequences_by_family.items():
    if len(records) < 2:
        print(f"[WARN] {family} hat weniger als 2 Sequenzen – übersprungen.")
        continue

    family_dir = os.path.join(output_dir, f"msa_{family}")
    os.makedirs(family_dir, exist_ok=True)

    input_path = os.path.join(family_dir, f"{family}.fasta")
    output_path = os.path.join(family_dir, f"{family}_aligned.fasta")

    with open(input_path, "w") as f:
        SeqIO.write(records, f, "fasta")

    cmd = ["clustalo", "-i", input_path, "-o", output_path]
    print(f"[RUN] {' '.join(cmd)}")
    try:
        subprocess.run(cmd, check=True)
        print(f"[INFO] MSA für {family} gespeichert: {output_path}")
    except subprocess.CalledProcessError as e:
        print(f"[ERROR] Fehler beim MSA für {family}")
        print(e)

[INFO] 9 Genfamilien gefunden.
[RUN] clustalo -i consensus_clustalo/msa_IGHV1/IGHV1.fasta -o consensus_clustalo/msa_IGHV1/IGHV1_aligned.fasta
[INFO] MSA für IGHV1 gespeichert: consensus_clustalo/msa_IGHV1/IGHV1_aligned.fasta
[RUN] clustalo -i consensus_clustalo/msa_IGHV4/IGHV4.fasta -o consensus_clustalo/msa_IGHV4/IGHV4_aligned.fasta
[INFO] MSA für IGHV4 gespeichert: consensus_clustalo/msa_IGHV4/IGHV4_aligned.fasta
[RUN] clustalo -i consensus_clustalo/msa_IGHV2/IGHV2.fasta -o consensus_clustalo/msa_IGHV2/IGHV2_aligned.fasta
[INFO] MSA für IGHV2 gespeichert: consensus_clustalo/msa_IGHV2/IGHV2_aligned.fasta
[RUN] clustalo -i consensus_clustalo/msa_IGHV3/IGHV3.fasta -o consensus_clustalo/msa_IGHV3/IGHV3_aligned.fasta
[INFO] MSA für IGHV3 gespeichert: consensus_clustalo/msa_IGHV3/IGHV3_aligned.fasta
[RUN] clustalo -i consensus_clustalo/msa_IGHV7/IGHV7.fasta -o consensus_clustalo/msa_IGHV7/IGHV7_aligned.fasta
[INFO] MSA für IGHV7 gespeichert: consensus_clustalo/msa_IGHV7/IGHV7_aligned.fasta

Es wird eine Datei erstellt, die die Genfamilien mit ihrer Consensussequenz auflistet. 

In [24]:
import os
import csv
from Bio import SeqIO
from Bio.Align import MultipleSeqAlignment, AlignInfo

# Verzeichnis mit den MSA-Dateien (angepasst an dein Output-Verzeichnis)
msa_root_dir = "consensus_clustalo"

# Ergebnisliste: tuples (family, consensus_sequence)
consensus_results = []

# Durchlaufe alle Familien-Unterordner
for family_dir in os.listdir(msa_root_dir):
    family_path = os.path.join(msa_root_dir, family_dir)
    if not os.path.isdir(family_path):
        continue

    # Annahme: Dateiname ist <family>_aligned.fasta
    family_name = family_dir.replace("msa_", "")
    msa_file = os.path.join(family_path, f"{family_name}_aligned.fasta")

    if not os.path.isfile(msa_file):
        print(f"[WARN] MSA-Datei nicht gefunden für {family_name}: {msa_file}")
        continue

    # MSA einlesen
    alignment = list(SeqIO.parse(msa_file, "fasta"))
    if len(alignment) == 0:
        print(f"[WARN] Leeres Alignment für {family_name}")
        continue

    msa = MultipleSeqAlignment(alignment)

    # Konsensus berechnen
    summary_align = AlignInfo.SummaryInfo(msa)
    consensus = summary_align.dumb_consensus()

    consensus_results.append((family_name, str(consensus)))

# Ergebnisse in CSV speichern
csv_path = os.path.join(msa_root_dir, "consensus_sequences.csv")
with open(csv_path, "w", newline="") as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(["family", "consensus_sequence"])
    writer.writerows(consensus_results)

print(f"[INFO] Konsensus-Sequenzen für {len(consensus_results)} Familien gespeichert in {csv_path}")

/opt/anaconda3/envs/project/lib/python3.10/site-packages/Bio/Align/AlignInfo.py:62: BiopythonDeprecationWarning: The `dumb_consensus` method is deprecated and will be removed in a future release of Biopython. As an alternative, you can convert the multiple sequence alignment object to a new-style Alignment object by via its `.alignment` property, and then create a Motif object. You can then use the `.consensus` or `.degenerate_consensus` property of the Motif object to get a consensus sequence. For more control over how the consensus sequence is calculated, you can call the `calculate_consensus` method on the `.counts` property of the Motif object. This is an example for a multiple sequence alignment `msa` of DNA nucleotides:
>>> from Bio.Seq import Seq
>>> from Bio.SeqRecord import SeqRecord
>>> from Bio.Align import MultipleSeqAlignment
>>> from Bio.Align.AlignInfo import SummaryInfo
>>> msa = MultipleSeqAlignment([SeqRecord(Seq('ACGT')),
...                             SeqRecord(Seq

[INFO] Konsensus-Sequenzen für 9 Familien gespeichert in consensus_clustalo/consensus_sequences.csv
